# LoRA Fine-Tuning on Text Data with a Small Model

In [ ]:
!pip install bitsandbytes accelerate loralib transformers datasets peft trl safetensors flash-linear-attention

## Load Pre-trained Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, GenerationConfig, pipeline

In [ ]:
from IPython.display import display_markdown, Latex, display
def print_markdown(content):
    if isinstance(content, str) and "\\" in content:
        display(Latex(content))
    else:
        display_markdown(content, raw=True)

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B-Base"
OUTPUT_DIR = "./lora-demo"

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Model: {MODEL_NAME}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if device in ("cuda", "mps") else torch.float32

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype
).to(device)

print(f"Tokenizer pad token: {tokenizer.pad_token}")

## Get baseline generation

In [ ]:
import copy
generation_config = copy.deepcopy(model.generation_config)

In [ ]:
generation_config.max_new_tokens = 500
generation_config.do_sample = True
generation_config.temperature = 0.5

In [ ]:
generator = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer
)

**Question:** Which viruses are best known to cause myocarditis?  
**Expected answer:** Enterovirus, Adenovirus and Coxsackie B viruses.  

In [ ]:
%%time
prompt = "Which viruses are best known to cause myocarditis?"
result = generator(prompt, generation_config=generation_config)

In [ ]:
print_markdown(result[0]["generated_text"])

## Load and prepare dataset

In [ ]:
import pandas as pd
import datasets

In [ ]:
bioasq_df = pd.read_csv("bioasq.csv")
bioasq_df.shape

In [ ]:
bioasq_df.sample(5)

In [ ]:
bioasq_df.drop(columns=["relevant_passage_ids"], inplace=True)

### Create a dataset object

In [ ]:
bioasq_dataset = datasets.Dataset.from_pandas(bioasq_df)

### Format dataset for fine-tuning

In [ ]:
def format_instruction(example):
    return {
        "text": f"Question: {example['question']}\nAnswer: {example['answer']}"
    }

In [ ]:
formatted_dataset = bioasq_dataset.map(format_instruction)

In [ ]:
print(formatted_dataset[42]["text"])

## Setting up LoRA

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

- **r:** The rank of the A and B matrices
- **lora_alpha:** Scaling factor. Usually 2x the rank.
- **target_modules:** The portions of the model we want to optimize with LoRA (depends on model architecture).
- **lora_dropout:** Probability of dropout to helps prevent the model from overfitting.
- **bias:** Whether to train LoRA on bias components as well. Set to `none` means we are training only on weighs component.
- **task_type:** Setting to `CAUSAL_LM` because the specific language model we're using is "causal".

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

### Attach LoRA adapter to the model

In [ ]:
model = get_peft_model(model, lora_config)

### Configure supervised fine-tuning

In [ ]:
training_args = SFTConfig(
    dataset_text_field="text",  
    max_length=512,
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=10,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=20,
    max_steps=100,
    save_strategy="epoch",
    optim="adamw_torch"
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_args,
    processing_class=tokenizer
)

### Total trainable parameters

In [ ]:
trainer.model.print_trainable_parameters()

### Total training steps (if not using `max_steps`)

In [ ]:
effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
total_steps = ((len(bioasq_dataset)+1) // effective_batch_size) * training_args.num_train_epochs

In [ ]:
print(f"Total training steps: {total_steps}")

## Training LoRA

In [ ]:
trainer.train()

## Save trained model (LoRA Adapter and Tokenizer)

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved LoRA adapter to:", OUTPUT_DIR)

## Reload base model and attach LoRA adapter

In [ ]:
from peft import PeftModel

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device in ("cuda", "mps") else torch.float32

In [ ]:
# Base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype
).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_generator = pipeline(
    "text-generation", 
    model=base_model, 
    tokenizer=tokenizer
)

# Base model + LoRA adapter
adapted_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
adapted_generator = pipeline(
    "text-generation",
    model=adapted_model,
    tokenizer=tokenizer,
    device=device
)

## Compare outputs of base and fine-tuned model

In [ ]:
import copy
generation_config = copy.deepcopy(model.generation_config)

In [ ]:
generation_config.max_new_tokens = 200
generation_config.do_sample = True
generation_config.temperature = 0.5

In [ ]:
questions = ["Which viruses are best known to cause myocarditis?",
             "What is the role of the blood-brain barrier?",
             "From which cell type is leptin secreted?",
             "What is the causative agent of the Panama disease affecting bananas?"
            ]

In [ ]:
for q in questions:
    print(f"\n{"="*150}")
    print(f"Question: {q}")

    base_out = base_generator(q, generation_config=generation_config)[0]["generated_text"]
    lora_out = adapted_generator(q, generation_config=generation_config)[0]["generated_text"]

    print("\n--- Base model ---")
    print(base_out)

    print("\n--- LoRA fine-tuned model ---")
    print(lora_out)

---